In [1]:
# Conditional Edges 조건 분기 - 질문 종류에 따라 그래프이 실행 경로를 동적으로 설정 가능
!pip install langgraph

In [3]:
from langgraph import graph
from typing import TypedDict, List
from langgraph.graph import StateGraph, END, START
from IPython.display import Image

class RouteState(TypedDict):
  query:str
  result:str

def math_node(state:RouteState) -> RouteState:
  q = state["query"]
  # 원래는 LLM으로 질문 종류를 파악해야 하나 편의상 ...
  answer = f"수학 관련 질문으로 분류됨 : {q}"
  return {"query":q, "result":answer}

def chat_node(state:RouteState) -> RouteState:
  q = state["query"]
  # 원래는 LLM으로 질문 종류를 파악해야 하나 편의상 ...
  answer = f"일반 대화로 분류됨 : {q}"
  return {"query":q, "result":answer}

def router_node(state:RouteState) -> RouteState:
  return state

def route_decision(state:RouteState) -> str:    # 분기 조건 함수
  q = state["query"]     # 영어 질문일 경우 .lower() 추가
  if any(ch.isdigit() for ch in q) or any(word in q for word in ["더하기", "빼기", "나누기"]):
    return "math"
  else:
    return "chat"

def build_graph():
  graph = StateGraph(RouteState)

  graph.add_node("router", router_node)
  graph.add_node("math_node", math_node)
  graph.add_node("chat_node", chat_node)

  graph.set_entry_point("router")

  # 분기
  graph.add_conditional_edges(
      "router",
      route_decision,
      {
          "math":"math_node",  # 'math_node'를 'math'로 변경
          "chat":"chat_node"    # 'chat_node'를 'chat'으로 변경
      }
  )

  graph.add_edge("math_node", END)
  graph.add_edge("chat_node", END)

  app = graph.compile()

  # 시각화 파일로 저장
  g = app.get_graph()
  png_bytes = g.draw_mermaid_png()     # Mermaid 기반 png 이미지 얻기

  with open("graph.png", "wb") as f:
    f.write(png_bytes)

  return app

if __name__=="__main__":
  graph = build_graph()

  queries = [
      "1 더하기 13은 얼마야?",
      "날씨가 추울 때 점심 메뉴 10가지 추천해줘.",
      "십 곱하기 십 계산해줘",
      "너를 소개해줘"
  ]

  for q in queries:
    print(f"\n질문은 {q}")
    final_state = graph.invoke({"query":q, "result":""})
    print(f"[최종 result] : {final_state["result"]}")


질문은 1 더하기 13은 얼마야?
[최종 result] : 수학 관련 질문으로 분류됨 : 1 더하기 13은 얼마야?

질문은 날씨가 추울 때 점심 메뉴 10가지 추천해줘.
[최종 result] : 수학 관련 질문으로 분류됨 : 날씨가 추울 때 점심 메뉴 10가지 추천해줘.

질문은 십 곱하기 십 계산해줘
[최종 result] : 일반 대화로 분류됨 : 십 곱하기 십 계산해줘

질문은 너를 소개해줘
[최종 result] : 일반 대화로 분류됨 : 너를 소개해줘
